# Ocean sounds: does synthetic data help when real recordings are scarce?

With all the real data, a classifier is already at 0.95 fine-grained F1, and 500 real clips per
class are almost as good: synthetic data has nothing to add there. The practical case for rare
species is the opposite: **only a few labelled recordings per class**. This notebook asks whether
adding generated clips helps a classifier trained on 16, 64 or 256 real clips per class.

**Two generators, two different questions:**

| Condition | Generator | Question it answers |
|---|---|---|
| Real only | none | the baseline at each budget |
| **Real + synthetic (same-data generator)** | Path A trained on **only the same N clips** (`ocean_scarce_generators.ipynb`; N = 64 and 256) | **The honest test:** with only N recordings, does generating more help? |
| Real + synthetic (full-data generator) | the existing ocean Path A, trained on all ~25k clips | **An upper bound:** if a good generator is available (e.g. trained on other sites or seasons), does it help a classifier with few labels? It has seen far more real data than the classifier, so it is not evidence about scarcity itself. |
| Synthetic only (same-data generator) | as above | how much the scarce generator's clips carry on their own |

**Fixed before running:**
* Budgets: 16, 64, 256 real clips per class, nested (the 16 are inside the 64, the 64 inside the
  256), the exact clips the scarce generators were trained on (`tstr.scarce_indices`, seed 0).
  No same-data generator exists for 16 (one would be needed per budget); that budget shows the
  upper-bound condition only.
* At each budget the recipe (learning rate, steps, SpecAugment) is chosen on **Real only**, fold 4,
  and reused unchanged for every condition at that budget. That favours the baseline, so any gain
  from synthetic data is conservative.
* Mixing: every training batch is **half real, half synthetic**. With 96 real clips against 3,000
  synthetic ones, plain pooling would make real data 3% of what the classifier sees; a fixed half
  keeps the real clips as the anchor at every budget.
* Classifier front end: the 10 Hz-1 kHz one, with its dB window fitted on the budget's real clips
  only.
* Headline: fine-grained F1 (three groupers + squirrelfish) on fold 5, **Real + synthetic (same-data)
  minus Real only**, with 95% grouped-bootstrap intervals. 3 seeds per condition.

Runtime on a T4: about 30-45 minutes. Needs the synthetic set from `ocean_synthetic_only.ipynb`
and, for the honest conditions, `ocean_scarce_generators.ipynb`; a missing generator is skipped with
a note. Every condition is saved when it finishes.

## 1. Setup

In [ ]:
!pip install -q diffusers accelerate soundfile
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time, subprocess
REPO_DIR = "/content/thesis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/1jvaldez1/audio-diffusion-synth.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import common as C
import tstr as T

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path

assert hasattr(T, "scarce_indices"), "old code: pull the repo, then restart the session"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
commit = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"]).decode().strip()

## 2. Config

In [ ]:
QUICK_TEST = False          # True: tiny run end to end, numbers meaningless

# data: must match notebooks/ocean_path_a.ipynb
SOURCE      = "/content/drive/MyDrive/thesis_runs/data/ocean_sounds_thesis.zip"
GROUP_SIZE  = 20
HIGHPASS_HZ = 20.0
NORMALIZE   = "rms"
OUT_ROOT    = "/content/drive/MyDrive/thesis_runs/ocean"

BUDGETS     = (16, 64, 256)
SUBSET_SEED = 0             # must match ocean_scarce_generators.ipynb
CLS_FE      = dict(n_fft=2048, f_min=10.0, f_max=1000.0)
FINE = ["Nassau grouper", "Yellowfin grouper", "Black grouper", "Squirrelfish"]

GRID = [dict(lr=lr, steps=st, specaug=sa) for lr in (1e-3, 3e-3) for st in (500, 1500) for sa in (False, True)]
BATCH, N_SEEDS, N_BOOT = 64, 3, 2000

cfg = C.ocean_config(out_root=OUT_ROOT, data_root=f"{OUT_ROOT}/data")
TAG = f"_hp{HIGHPASS_HZ:g}_{NORMALIZE}_bal"
FULL_GEN_WAVE = f"{cfg.out_root}/synonly/path_a128{TAG}_n500_g1/gen_wave_gl64.pt"
SCARCE_ROOT = f"{cfg.out_root}/scarce"
if QUICK_TEST:
    BUDGETS, N_SEEDS, N_BOOT = (2, 4), 2, 200
    GRID = [dict(lr=3e-3, steps=20, specaug=False), dict(lr=3e-3, steps=20, specaug=True)]
    FULL_GEN_WAVE = FULL_GEN_WAVE.replace("_n500_g1", "_n4_g1_quick").replace("gl64", "gl8")
    SCARCE_ROOT += "_quick"
def scarce_wave(n):
    p = Path(SCARCE_ROOT) / f"path_a128{TAG}_n{n}" / f"gen_wave_gl{8 if QUICK_TEST else 64}.pt"
    return p if p.exists() else None

OUT_DIR = Path(f"{cfg.out_root}/scarcity" + ("_quick" if QUICK_TEST else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)
C.seed_everything(cfg.seed)
print("full-data synthetic set:", FULL_GEN_WAVE, "(found)" if os.path.exists(FULL_GEN_WAVE) else "(MISSING)")
for n in BUDGETS:
    print(f"same-data generator, {n} per class:", scarce_wave(n) or "not found (condition skipped)")

## 3. Data

In [ ]:
data = C.load_ocean(cfg, SOURCE, group_size=GROUP_SIZE, highpass_hz=HIGHPASS_HZ, normalize=NORMALIZE)
class_names = data["class_names"]; n_classes = len(class_names)
train_split, val_split, test_split = C.split_by_fold(data, cfg)
del data
fine_idx = [class_names.index(n) for n in FINE]

def center_crop(w):
    off = (w.shape[-1] - cfg.clip_samples) // 2
    return w[..., off:off + cfg.clip_samples]

val_wave, val_y = center_crop(val_split["waveforms"]), val_split["labels"].numpy()
test_wave, test_y = center_crop(test_split["waveforms"]), test_split["labels"].numpy()
test_groups = np.array(test_split["groups"])

budget_idx = {n: T.scarce_indices(train_split["labels"].numpy(), n, seed=SUBSET_SEED) for n in BUDGETS}
for a, b in zip(BUDGETS, BUDGETS[1:]):
    assert set(budget_idx[a]) <= set(budget_idx[b]), "budgets are not nested"
for n in BUDGETS:
    f = Path(SCARCE_ROOT) / f"path_a128{TAG}_n{n}" / "train_indices.npy"
    if f.exists():
        assert np.array_equal(np.load(f), budget_idx[n]), f"the {n}-clip generator was trained on different clips"

full_gen = torch.load(FULL_GEN_WAVE)
full_gen_wave = full_gen if torch.is_tensor(full_gen) else full_gen["waves"]
n_full = len(full_gen_wave) // n_classes
full_gen_y = torch.arange(n_classes).repeat_interleave(n_full)
print(f"budgets {BUDGETS} (nested) | full-data synthetic {tuple(full_gen_wave.shape)} | val {len(val_y)} | test {len(test_y)}")

## 4. The classifier and the conditions

The probe network (`tstr.ProbeCNN`) and training loop (`tstr.train_clf`) used in every TSTR
experiment, unchanged; the architecture table is in `ocean_synthetic_only.ipynb`. What is new
here is only the training data and how it is mixed:

| Choice | Reason |
|---|---|
| Half real, half synthetic in every batch (a batch of 64: 32 + 32) | Keeps the real clips the anchor at every budget. Pooling would let 3,000 synthetic clips drown 96 real ones, and the result would mostly measure synthetic-only training. |
| The same step count for real-only and mixed training | Both see the same number of batches, so a difference is not more training in disguise. |
| Front end per budget, dB window fitted on that budget's real clips | At 16 clips per class even the level calibration should not come from data the classifier does not have. |

In [ ]:
def f1_per_class(y, p, k=n_classes):
    cm = np.bincount(y * k + p, minlength=k * k).reshape(k, k)
    tp = np.diag(cm).astype(float)
    prec, rec = tp / np.maximum(cm.sum(0), 1), tp / np.maximum(cm.sum(1), 1)
    return np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0.0)

@torch.no_grad()
def predict(model, Xs, batch=256):
    return torch.cat([model(Xs[i:i + batch].float()).float().softmax(-1).cpu() for i in range(0, len(Xs), batch)]).numpy()

def mixed(get_real, get_syn):
    def get(g, n):
        xr, yr = get_real(g, n // 2); xs, ys = get_syn(g, n - n // 2)
        return torch.cat([xr, xs]), torch.cat([yr, ys])
    return get

cls_cfg = C.ocean_config(out_root=OUT_ROOT, data_root=f"{OUT_ROOT}/data", **CLS_FE)

## 5. Run every budget

For each budget: fit the front end on its real clips, choose the recipe on Real only (fold 4),
then train every condition with that recipe and score fold 5. Saved per condition.

In [ ]:
probs, recipes = {}, {}
t0 = time.time()
for n in BUDGETS:
    idx = torch.from_numpy(budget_idx[n])
    real_wave, real_y = center_crop(train_split["waveforms"][idx]).float(), train_split["labels"][idx]
    fe = C.MelFrontEnd(cls_cfg, device=device).fit(real_wave)
    spec = lambda w, dt=torch.float32: torch.cat([fe(w[i:i + 256].to(device).float()).to(dt) for i in range(0, len(w), 256)])
    X_val, X_test = spec(val_wave, torch.float16), spec(test_wave, torch.float16)
    get_real = T.batch_fixed(spec(real_wave), real_y, device)
    conds = {"Real only": get_real,
             "Real + synthetic (full-data generator)": mixed(get_real, T.batch_fixed(spec(full_gen_wave), full_gen_y, device))}
    sw = scarce_wave(n)
    if sw is not None:
        d = torch.load(sw)
        get_syn = T.batch_fixed(spec(d["waves"]), d["labels"].long(), device)
        conds["Real + synthetic (same-data generator)"] = mixed(get_real, get_syn)
        conds["Synthetic only (same-data generator)"] = get_syn

    rec_file = OUT_DIR / f"recipe_n{n}.json"
    if rec_file.exists():
        recipes[n] = json.load(open(rec_file))["recipe"]
    else:
        rows = []
        for rc in GRID:
            m = T.train_clf(get_real, n_classes, seed=1000, device=device, batch=BATCH, **rc)
            f1 = f1_per_class(val_y, predict(m, X_val).argmax(1))
            rows.append({**rc, "fold-4 macro-F1": f1.mean()})
        grid = pd.DataFrame(rows)
        best = grid.sort_values(["fold-4 macro-F1", "steps", "specaug", "lr"], ascending=[False, True, True, True]).iloc[0]
        recipes[n] = dict(lr=float(best["lr"]), steps=int(best["steps"]), specaug=bool(best["specaug"]))
        json.dump({"recipe": recipes[n], "grid": grid.to_dict("records")}, open(rec_file, "w"), indent=2, default=float)
    print(f"\n=== {n} real clips per class: recipe {recipes[n]}")

    for name, get in conds.items():
        f = OUT_DIR / f"probs_n{n}_{name.replace(' ', '_').replace('(', '').replace(')', '')}.npz"
        if f.exists():
            z = np.load(f); ps = [z[k] for k in sorted(z.files)]
        else:
            ps = [predict(T.train_clf(get, n_classes, seed=s, device=device, batch=BATCH, **recipes[n]), X_test) for s in range(N_SEEDS)]
            np.savez(f, **{f"seed{s}": p for s, p in enumerate(ps)})
        probs[(n, name)] = ps
        f1s = np.array([f1_per_class(test_y, p.argmax(1)) for p in ps])
        print(f"   {name:<42} fine-grained {f1s[:, fine_idx].mean(1).mean():.3f} +/- {f1s[:, fine_idx].mean(1).std():.3f} "
              f"| macro {f1s.mean(1).mean():.3f}   ({(time.time() - t0) / 60:.1f} min)")

## 6. Results

The table gives the mean over seeds of each seed's F1. The differences below use the three seeds'
averaged predictions (an ensemble), resampled by test recording group, so a difference can be a
little larger or smaller than the gap between two rows of the table.

In [ ]:
rows = []
for (n, name), ps in probs.items():
    f1s = np.array([f1_per_class(test_y, p.argmax(1)) for p in ps])
    rows.append({"budget": n, "condition": name, "fine-grained F1": f1s[:, fine_idx].mean(1).mean(),
                 "fine sd": f1s[:, fine_idx].mean(1).std(), "macro-F1": f1s.mean(1).mean(),
                 **{c: f1s[:, i].mean() for i, c in enumerate(class_names)}})
results = pd.DataFrame(rows).set_index(["budget", "condition"])
display(results.round(3))

# paired grouped bootstrap: each condition vs Real only at the same budget
rng = np.random.default_rng(0)
cls_groups = [[np.flatnonzero((test_y == c) & (test_groups == g)) for g in np.unique(test_groups[test_y == c])] for c in range(n_classes)]
pred = {k: np.mean(ps, axis=0).argmax(1) for k, ps in probs.items()}
boot = {k: [] for k in pred}
for _ in range(N_BOOT):
    idx = np.concatenate([np.concatenate([grp[j] for j in rng.integers(len(grp), size=len(grp))]) for grp in cls_groups])
    for k, p in pred.items():
        boot[k].append(f1_per_class(test_y[idx], p[idx])[fine_idx].mean())
boot = {k: np.array(v) for k, v in boot.items()}
diff_rows = []
for (n, name) in probs:
    if name == "Real only":
        continue
    d = boot[(n, name)] - boot[(n, "Real only")]
    lo, hi = np.percentile(d, [2.5, 97.5])
    diff_rows.append({"budget": n, "condition": name, "fine-grained F1 vs Real only": d.mean(), "95% CI": (round(lo, 3), round(hi, 3)),
                      "verdict": "helps" if lo > 0 else ("hurts" if hi < 0 else "no clear difference")})
diffs = pd.DataFrame(diff_rows).set_index(["budget", "condition"])
diffs.round(3)

In [ ]:
ref = None
for f in sorted(Path(f"{cfg.out_root}/synonly").glob(f"path_a128{TAG}_n500_g1/synonly_*.json")):
    ref = {r["training set"]: r["fine-grained F1"] for r in json.load(open(f))["results"]}
fig, ax = plt.subplots(figsize=(7, 4.2))
styles = {"Real only": ("#555555", "o"), "Real + synthetic (same-data generator)": ("#1b9e77", "s"),
          "Real + synthetic (full-data generator)": ("#3b6ea8", "^"), "Synthetic only (same-data generator)": ("#d95f02", "x")}
for name, (col, mk) in styles.items():
    pts = [(n, results.loc[(n, name), "fine-grained F1"], results.loc[(n, name), "fine sd"]) for n in BUDGETS if (n, name) in results.index]
    if pts:
        x, y, e = zip(*pts)
        ax.errorbar(x, y, yerr=e, color=col, marker=mk, capsize=3, label=name)
if ref:
    ax.axhline(ref["Real-full"], color="k", ls="--", lw=0.8, label=f"all real data ({ref['Real-full']:.3f})")
ax.set_xscale("log", base=2); ax.set_xticks(BUDGETS, [str(b) for b in BUDGETS])
ax.set_xlabel("real clips per class"); ax.set_ylabel("fine-grained F1, real fold 5"); ax.legend(fontsize=7)
ax.set_title("does synthetic data help when real recordings are scarce?")
plt.tight_layout(); plt.savefig(OUT_DIR / "scarcity.png", dpi=130); plt.show()

## 7. Save

In [ ]:
out = {"commit": commit, "quick_test": QUICK_TEST, "budgets": BUDGETS, "recipes": {str(k): v for k, v in recipes.items()},
       "results": results.round(4).reset_index().to_dict("records"),
       "differences": diffs.reset_index().astype(str).to_dict("records"),
       "missing_same_data_generators": [n for n in BUDGETS if scarce_wave(n) is None]}
with open(OUT_DIR / f"scarcity_{commit}.json", "w") as f:
    json.dump(out, f, indent=2, default=float)
print("saved", OUT_DIR / f"scarcity_{commit}.json")

## How to read the result

* **The headline is the same-data rows of the differences table.** "Helps" at a budget means a
  generator trained on only those N recordings made the classifier better than those N recordings
  alone, beyond the test-set noise. That is the claim synthetic data for rare species needs.
* **The full-data rows are an upper bound,** not evidence about scarcity: that generator learned
  from ~25k clips the classifier never had. A large gain there and none for the same-data
  generator means the benefit comes from the generator's extra data, not from generation itself.
* **Read the same-data rows together with the memorisation check** in
  `ocean_scarce_generators.ipynb`: a generator that copies its training clips gives "synthetic"
  data that is really augmented real data.
* **The gap closes with more real data.** Expect any benefit to be largest at 16 or 64 clips per
  class and to vanish by 256, where real-only is already close to the full-data result.